In [ ]:
!gdown 1GNFGWfUbgXfELx5fZtjTjU2qqWnEa-Lr

In [ ]:
!unzip /kaggle/working/Flickr24K.zip
!rm /kaggle/working/Flickr24K.zip

In [ ]:
# 0) Проверяем, что датасет FlareX подключён (Add Input -> flarex)
import subprocess
print(subprocess.run(["find", "/kaggle/input", "-maxdepth", "4"], capture_output=True, text=True).stdout)

In [ ]:
# 1) Находим реальные пути к Flare2D/input (блики) и Flare2D/gt (источники света)
import glob, os

candidates_input = glob.glob("/kaggle/input/**/Flare2D/input", recursive=True)
candidates_gt = glob.glob("/kaggle/input/**/Flare2D/gt", recursive=True)

assert candidates_input, "Не найдена папка Flare2D/input — проверь, что датасет 'flarex' подключён (Add Input)."
assert candidates_gt, "Не найдена папка Flare2D/gt — проверь, что датасет 'flarex' подключён (Add Input)."

FLARE_INPUT_DIR = candidates_input[0]
FLARE_GT_DIR = candidates_gt[0]
print("FLARE_INPUT_DIR =", FLARE_INPUT_DIR, "(", len(os.listdir(FLARE_INPUT_DIR)), "файлов )")
print("FLARE_GT_DIR    =", FLARE_GT_DIR, "(", len(os.listdir(FLARE_GT_DIR)), "файлов )")

In [ ]:
# 2) Ищем папку с фоновыми кадрами (любой подключённый датасет, кроме flarex).
#    Берём папку с наибольшим числом jpg/png файлов среди прочих Input-датасетов.
import glob, os
from collections import defaultdict

counts = defaultdict(int)
for root, dirs, files in os.walk("/kaggle/input"):
    if "flarex" in root.lower() or "FlareX" in root:
        continue
    n = sum(1 for f in files if f.lower().endswith((".jpg", ".jpeg", ".png")))
    if n > 0:
        counts[root] += n

assert counts, ("Не нашёл фоновых кадров ни в одном датасете, кроме flarex. "
                "Подключи датасет с кадрами вождения через Add Input, например "
                "'sshikamaru/udacity-self-driving-car-dataset'.")

BG_DIR = max(counts, key=counts.get)
bg_files = [os.path.join(BG_DIR, f) for f in os.listdir(BG_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
print("BG_DIR =", BG_DIR, "-", len(bg_files), "кадров")

In [ ]:
# 3) Автономная версия Paired_Flare_Image_Loader из FlareX/basicsr/data/flareX_dataset.py
#    (скопировано без зависимости от пакета basicsr, чтобы не собирать CUDA-расширения DCN)
import torch
import torch.utils.data as data
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import math
import numpy as np
from PIL import Image
import glob
import random
from torch.distributions import Normal


class RandomGammaCorrection(object):
    def __init__(self, gamma=None):
        self.gamma = gamma

    def __call__(self, image):
        if self.gamma is None:
            gammas = [0.5, 1, 2]
            self.gamma = random.choice(gammas)
            return TF.adjust_gamma(image, self.gamma, gain=1)
        elif isinstance(self.gamma, tuple):
            gamma = random.uniform(*self.gamma)
            return TF.adjust_gamma(image, gamma, gain=1)
        elif self.gamma == 0:
            return image
        else:
            return TF.adjust_gamma(image, self.gamma, gain=1)


def remove_background(image):
    # чистая torch-версия (без numpy) — работает как на CPU, так и на GPU-тензорах
    _EPS = 1e-7
    rgb_max = image.amax(dim=(0, 1), keepdim=True)
    rgb_min = image.amin(dim=(0, 1), keepdim=True)
    image = (image - rgb_min) * rgb_max / (rgb_max - rgb_min + _EPS)
    return image


class SimpleFlareLoader(data.Dataset):
    """Урезанная версия Paired_Flare_Image_Loader: без depth-aware (3D) веток,
    без пар lq/gt (для этого нужен отдельный датасет), только random-синтез
    блика поверх фонового фото — то, что нужно для 'нагенерить бликов'."""

    def __init__(self, background_dir, flare_dir, light_dir, img_size=512, device="cpu"):
        self.ext = ["png", "jpeg", "jpg", "bmp", "tif"]
        self.data_list = []
        [self.data_list.extend(glob.glob(background_dir + "/*." + e)) for e in self.ext]
        assert len(self.data_list) > 0, f"Нет фоновых фото в {background_dir}"

        self.flare_list = sorted(sum([glob.glob(flare_dir + "/*." + e) for e in self.ext], []))
        self.light_list = sorted(sum([glob.glob(light_dir + "/*." + e) for e in self.ext], []))
        assert len(self.flare_list) > 0, f"Нет изображений бликов в {flare_dir}"
        assert len(self.flare_list) == len(self.light_list), "Число flare и light изображений не совпадает"

        self.img_size = img_size
        self.device = device
        self.to_tensor = transforms.ToTensor()

        self.transform_base = transforms.Compose([
            transforms.RandomCrop((img_size, img_size), pad_if_needed=True, padding_mode="reflect"),
            transforms.RandomHorizontalFlip(),
            # RandomVerticalFlip() тут нарочно убран: это фон с дороги (небо сверху, асфальт снизу),
            # вертикальный флип переворачивал сцену вверх ногами в ~50% кадров.
        ])

        self.transform_flare = transforms.Compose([
            transforms.RandomAffine(degrees=(0, 0), scale=(0.8, 1.5),
                                     translate=(300 / 1440, 300 / 1440), shear=(-20, 20)),
            transforms.CenterCrop((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
        ])

        self.blur_transform = transforms.GaussianBlur(21, sigma=(0.1, 3.0))

    def __len__(self):
        return len(self.data_list)

    @torch.no_grad()
    def __getitem__(self, index):
        gamma = np.random.uniform(1.8, 2.2)
        adjust_gamma = RandomGammaCorrection(gamma)
        adjust_gamma_reverse = RandomGammaCorrection(1 / gamma)

        # декодирование JPEG/PNG остаётся на CPU (PIL), но сразу после to_tensor
        # переносим на GPU — вся дальнейшая математика (аффин, блюр, гамма) идёт там
        base_img = Image.open(self.data_list[index]).convert("RGB")
        base_img = self.to_tensor(base_img).to(self.device, non_blocking=True)
        base_img = adjust_gamma(base_img)
        base_img = self.transform_base(base_img)

        sigma_chi = 0.01 * np.random.chisquare(df=1)
        base_img = Normal(base_img, sigma_chi).sample()
        gain = np.random.uniform(0.5, 1.2)
        base_img = torch.clamp(gain * base_img, min=0, max=1)

        choice_index = random.randint(0, len(self.flare_list) - 1)
        flare_img = self.to_tensor(Image.open(self.flare_list[choice_index]).convert("RGB")).to(self.device, non_blocking=True)
        flare_img = adjust_gamma(flare_img)
        light_img = self.to_tensor(Image.open(self.light_list[choice_index]).convert("RGB")).to(self.device, non_blocking=True)
        light_img = adjust_gamma(light_img)

        flare_img = remove_background(flare_img)

        flare_merge = torch.cat((flare_img, light_img), dim=0)
        flare_merge = self.transform_flare(flare_merge)
        flare_img, light_img = torch.split(flare_merge, 3, dim=0)

        flare_img = torch.clamp(self.blur_transform(flare_img), min=0, max=1)
        light_img = torch.clamp(self.blur_transform(light_img), min=0, max=1)

        merge_img = torch.clamp(base_img + flare_img, min=0, max=1)
        gt_img = torch.clamp(base_img + light_img, min=0, max=1)
        flare_only = torch.clamp(flare_img - light_img, min=0, max=1)

        return {
            "gt": adjust_gamma_reverse(gt_img),
            "flare": adjust_gamma_reverse(flare_only),
            "lq": adjust_gamma_reverse(merge_img),
        }

In [ ]:
# 3b) Включаем GPU, если он подключён в ноутбуке (Settings -> Accelerator -> GPU T4/P100).
#     Вся тяжёлая математика (блюр, аффинные преобразования блика) пойдёт на CUDA.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True  # размер картинок фиксирован (img_size), это ускоряет conv/blur
print("Используем устройство:", DEVICE, "-", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "GPU не найден (Settings -> Accelerator -> GPU)")

In [ ]:
# 4) Генерируем N примеров и сохраняем на диск.
#    В архив идут ТОЛЬКО кадры с бликом (lq) — без gt/flare-исходников.
import os
from torchvision.utils import save_image
from tqdm.notebook import tqdm

N = 1000  # сколько картинок сгенерировать
OUT_DIR = "/kaggle/working/output"
os.makedirs(OUT_DIR, exist_ok=True)

dataset = SimpleFlareLoader(background_dir=BG_DIR, flare_dir=FLARE_INPUT_DIR, light_dir=FLARE_GT_DIR,
                             img_size=720, device=DEVICE)  # 720 = высота кадров BDD100K, без паддинга
print(f"Фонов: {len(dataset)}, бликов: {len(dataset.flare_list)}")

for i in tqdm(range(N)):
    idx = random.randint(0, len(dataset) - 1)
    sample = dataset[idx]
    save_image(sample["lq"], os.path.join(OUT_DIR, f"{i:04d}.png"))  # save_image сам перекинет тензор на CPU
    if (i + 1) % 100 == 0:
        print(f"{i + 1}/{N}")

print("Готово. Файлы в", OUT_DIR)

In [ ]:
# 5) Быстрый просмотр результата (превью не сохраняется в архив)
import matplotlib.pyplot as plt
from PIL import Image as PILImage

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(PILImage.open(f"{OUT_DIR}/{i:04d}.png"))
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Центральный кроп всех сгенерированных картинок
import os
from PIL import Image

CROP_SIZE = 512          # какой размер кропа нужен
SRC_DIR = OUT_DIR        # папка с исходными картинками (та, что уже есть)
DST_DIR = "/kaggle/working/output_cropped"
os.makedirs(DST_DIR, exist_ok=True)

files = sorted(f for f in os.listdir(SRC_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg")))

for name in files:
    img = Image.open(os.path.join(SRC_DIR, name))
    w, h = img.size
    left = (w - CROP_SIZE) // 2
    top = (h - CROP_SIZE) // 2
    cropped = img.crop((left, top, left + CROP_SIZE, top + CROP_SIZE))
    cropped.save(os.path.join(DST_DIR, name))

print(f"Готово: {len(files)} картинок обрезано до {CROP_SIZE}x{CROP_SIZE} -> {DST_DIR}")

In [ ]:
# 6) Упаковываем в zip, чтобы скачать через "Output" вкладку ноутбука
import shutil
shutil.make_archive("/kaggle/working/flarex_generated", "zip", DST_DIR)
print("Архив готов: /kaggle/working/flarex_generated.zip (скачать во вкладке Output/Data справа)")